# Where RSM's decisions read, and what reading the summary is worth

The attention declaration of the study protocol, on the confirmation rosters of Dark Key-to-Door (500 calls) and CountRecallMedium (one stream), frozen 8M endpoints, seeds 42 / 100 / 2026. Every decision of the summary carrier attends jointly to the four summary slots (its long-term memory), the earlier records of the 32-record buffer (its short-term memory) and its own record; the three masses sum to one.

- **(a) Where the decision reads.** RSM's attention mass on the summary by position in the segment, one line per block, beside the w/o-memory policy's mass on its READ slots (which always hold the initial memory, so any attention they draw comes from position alone) and the uniform share M/(M+i).
- **(b) What reading is worth.** The primary metric when an additive bias −β lowers the decision rows' scores on the summary keys (or, as the companion, on the earlier buffer keys); β = ∞ blocks them. The writer still reads the previous summary, so information can be carried but not read. The dashed line is the w/o-memory policy.
- **(c) The declared attention readings per block.** A1: RSM − w/o memory, summary mass over positions 2–32. A2 (CountRecall): queries whose evidence lives only in the summary − queries whose evidence is in the buffer. A3 (Key-to-Door): the first eight steps of later attempts − attempt 1's. A2 and A3 are position-adjusted (the seed's mean at each segment and position subtracted). Joint seed/task 95 % intervals.

Attention weights do not by themselves explain a decision; panel (b) is the causal reading, (a) and (c) where the decisions look. The numbers are the files `scripts/attention_read.py read` writes under `reports/<protocol>/confirmation/attention/` of the 8M study root; this notebook only draws them.

In [ ]:
"""Locate the repository, the study roots and the output folders.

Environment overrides, for a copy of the records elsewhere:
  REASONED_ICRL_OUTPUTS   study roots (default <repo>/outputs)
  FIGURE_OUTPUT_ROOT      where the figure is written (default notebooks/outputs/figures);
                          its tables and notes go to the sibling data/ folder
"""
import os
import sys
from pathlib import Path

from IPython.display import Markdown, display

REPO = next(
    p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").is_file()
)
sys.path.insert(0, str(REPO / "notebooks"))

from figures_common import default_layout, markdown_preview  # noqa: E402

OUTPUTS = Path(os.environ.get("REASONED_ICRL_OUTPUTS", REPO / "outputs"))
FIGURES_DIR = Path(
    os.environ.get("FIGURE_OUTPUT_ROOT", REPO / "notebooks/outputs/figures")
)
DATA_DIR = FIGURES_DIR.parent / "data"
layout = default_layout(REPO, outputs=OUTPUTS)
print(f"records under {OUTPUTS}\nfigures to  {FIGURES_DIR}\ntables to   {DATA_DIR}")

In [ ]:
import csv
import math
from collections import defaultdict
from collections.abc import Sequence
from dataclasses import dataclass
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from figures_common import (
    FULL_WIDTH,
    INK,
    INK_MUTED,
    METHOD_COLOUR,
    METHOD_CONDITION,
    PANEL,
    BuiltFigure,
    StudyLayout,
    paper_style,
    standalone,
    style_axes,
    style_of,
)
from matplotlib.axes import Axes
from matplotlib.lines import Line2D

ATTENTION_ENVIRONMENTS: tuple[str, ...] = ("keydoor", "countrecall")
"""The two benchmarks of the attention declaration."""
REFERENCE_CONDITION = "raw_segment"
BUFFER_COLOUR = "#0072B2"
"""Okabe-Ito blue: the buffer-read companion of panel (b)."""
LAYER_SHADES: tuple[str, ...] = ("#F4A582", METHOD_COLOUR, "#7F2704")
REFERENCE_SHADES: tuple[str, ...] = ("#D0D0D0", "#9A9A9A", "#5A5A5A")
BETAS: tuple[float, ...] = (0.0, 1.0, 2.0, 4.0, 8.0, math.inf)
TITLES = {"keydoor": "Dark Key-to-Door", "countrecall": "CountRecall"}
METRICS = {"keydoor": "Doors in 500 calls", "countrecall": "Exact accuracy"}
CONTRASTS = {
    "keydoor": ("A3", "later-attempt starts"),
    "countrecall": ("A2", "evidence only in the summary"),
}

Row = dict[str, object]


@dataclass(frozen=True)
class AttentionOptions:
    """``split`` names the roster the attention files were written for."""

    environments: tuple[str, ...] = ATTENTION_ENVIRONMENTS
    split: str = "confirmation"


def attention_directory(layout: StudyLayout, environment: str, split: str) -> Path:
    """``<8M root>/reports/<protocol>/<split>/attention`` of one benchmark."""
    root = layout.roots[(environment, METHOD_CONDITION)]
    return root / "reports" / layout.spec(environment).protocol / split / "attention"


def read_rows(path: Path) -> list[dict[str, str]]:
    if not path.is_file():
        return []
    with path.open(newline="", encoding="utf-8") as stream:
        return list(csv.DictReader(stream))


def number(text: object) -> float:
    return math.inf if str(text) == "inf" else float(str(text))


def profile_means(
    rows: Sequence[dict[str, str]],
) -> dict[tuple[str, int], list[tuple[int, float, float]]]:
    """Seed-mean summary mass by (cell, block): (position, mass, uniform share)."""
    cells: dict[tuple[str, int, int], list[float]] = defaultdict(list)
    uniform: dict[int, float] = {}
    for row in rows:
        key = (row["cell"], int(row["layer"]), int(row["position"]))
        cells[key].append(float(row["summary"]))
        uniform[int(row["position"])] = float(row["uniform_summary"])
    out: dict[tuple[str, int], list[tuple[int, float, float]]] = defaultdict(list)
    for (cell, layer, position), values in sorted(cells.items()):
        out[(cell, layer)].append((position, float(np.mean(values)), uniform[position]))
    return dict(out)


def draw_profile(ax: Axes, environment: str, rows: Sequence[dict[str, str]]) -> None:
    means = profile_means(rows)
    uniform: list[tuple[int, float]] = []
    for (cell, layer), points in sorted(means.items()):
        xs = [p for p, _, _ in points]
        ys = [m for _, m, _ in points]
        uniform = [(p, u) for p, _, u in points]
        if cell == METHOD_CONDITION:
            ax.plot(
                xs, ys, color=LAYER_SHADES[layer % 3], lw=1.4, solid_capstyle="round"
            )
        elif cell == REFERENCE_CONDITION:
            ax.plot(xs, ys, color=REFERENCE_SHADES[layer % 3], lw=1.0, ls="--")
    if uniform:
        ax.plot(*zip(*uniform, strict=True), color=INK, lw=0.8, ls=":")
    ax.set_xlim(1, 32)
    ax.set_ylim(0, 1)
    ax.set_xticks([1, 8, 16, 24, 32])
    ax.set_xlabel("Position of the decision in its segment")
    ax.set_ylabel(f"{TITLES[environment]}\nattention on the summary")
    ax.set_title("(a) where decisions read", loc="left")
    style_axes(ax, grid_axis="both")


def draw_dose(
    ax: Axes, environment: str, rows: Sequence[dict[str, str]], reference: float | None
) -> None:
    positions = {beta: index for index, beta in enumerate(BETAS)}
    retained = [r for r in rows if r["target"] == "none"]
    for target, colour, label in (
        ("summary", METHOD_COLOUR, "summary"),
        ("buffer", BUFFER_COLOUR, "buffer"),
    ):
        chosen = sorted(
            [r for r in rows if r["target"] == target] + retained,
            key=lambda r: number(r["beta"]),
        )
        xs = [
            positions[number(r["beta"])]
            for r in chosen
            if number(r["beta"]) in positions
        ]
        ys = [float(r["primary"]) for r in chosen if number(r["beta"]) in positions]
        low = [
            float(r["primary"]) - float(r["primary_lower"])
            for r in chosen
            if number(r["beta"]) in positions
        ]
        high = [
            float(r["primary_upper"]) - float(r["primary"])
            for r in chosen
            if number(r["beta"]) in positions
        ]
        if xs:
            ax.errorbar(
                xs,
                ys,
                yerr=[low, high],
                color=colour,
                lw=1.4 if target == "summary" else 1.0,
                capsize=2,
                label=label,
            )
    if reference is not None:
        ax.axhline(
            reference, color=style_of(REFERENCE_CONDITION).color, lw=1.0, ls="--"
        )
    ax.set_xticks(range(len(BETAS)))
    ax.set_xticklabels(["0", "1", "2", "4", "8", "∞"])
    ax.set_xlabel("Read bias β on the decision rows")
    ax.set_ylabel(METRICS[environment])
    ax.set_ylim(bottom=0)
    ax.set_title("(b) what reading the memory is worth", loc="left")
    style_axes(ax)


def draw_readings(ax: Axes, environment: str, rows: Sequence[dict[str, str]]) -> None:
    code, _ = CONTRASTS[environment]
    series = (("A1", METHOD_COLOUR, "o", -0.12), (code, INK, "s", 0.12))
    layers: set[int] = set()
    for prefix, colour, marker, shift in series:
        for row in rows:
            if not row["reading"].startswith(prefix) or row.get("layer", "") == "":
                continue
            layer = int(row["layer"])
            layers.add(layer)
            estimate = float(row["estimate"])
            ax.errorbar(
                estimate,
                layer + shift,
                xerr=[
                    [estimate - float(row["lower"])],
                    [float(row["upper"]) - estimate],
                ],
                color=colour,
                marker=marker,
                ms=4,
                lw=1.0,
                capsize=2,
            )
    ax.axvline(0.0, color=INK_MUTED, lw=0.8)
    ticks = sorted(layers) or [0, 1, 2]
    ax.set_yticks(ticks)
    ax.set_yticklabels([f"Block {t + 1}" for t in ticks])
    ax.invert_yaxis()
    ax.set_xlabel("Difference in attention on the summary")
    ax.set_title(f"(c) declared readings ({CONTRASTS[environment][0]})", loc="left")
    style_axes(ax, grid_axis="x")


def legend_items(environment: str) -> list[Line2D]:
    return [
        *(
            Line2D([], [], color=c, lw=1.4, label=f"RSM, block {i + 1}")
            for i, c in enumerate(LAYER_SHADES)
        ),
        Line2D(
            [],
            [],
            color=REFERENCE_SHADES[1],
            lw=1.0,
            ls="--",
            label="w/o memory: READ mass (a), score (b)",
        ),
        Line2D([], [], color=INK, lw=0.8, ls=":", label="uniform share M/(M+i)"),
        Line2D(
            [],
            [],
            color=METHOD_COLOUR,
            lw=1.4,
            marker="|",
            label="bias on summary keys",
        ),
        Line2D(
            [], [], color=BUFFER_COLOUR, lw=1.0, marker="|", label="bias on buffer keys"
        ),
        Line2D(
            [],
            [],
            color=METHOD_COLOUR,
            marker="o",
            ls="",
            label="A1: RSM \u2212 w/o memory",
        ),
        Line2D(
            [], [], color=INK, marker="s", ls="", label="A2 / A3: evidence in summary"
        ),
    ]


def reference_level(rows: Sequence[dict[str, str]]) -> float | None:
    for row in rows:
        if row["reading"].startswith("w/o memory primary"):
            return float(row["estimate"])
    return None


def build_attention_figure(
    layout: StudyLayout, options: AttentionOptions
) -> BuiltFigure:
    """The attention figure: one row per benchmark, panels (a)-(c)."""
    data: dict[str, dict[str, list[dict[str, str]]]] = {}
    sources: list[str] = []
    notes: dict[str, object] = {"split": options.split}
    for environment in options.environments:
        directory = attention_directory(layout, environment, options.split)
        data[environment] = {
            name: read_rows(directory / f"{name}.csv")
            for name in ("profile", "dose", "readings")
        }
        present = [name for name, rows in data[environment].items() if rows]
        notes[environment] = {
            "directory": layout.relative(directory),
            "present": present,
            "pending": [n for n in ("profile", "dose", "readings") if n not in present],
        }
        sources += [layout.relative(directory / f"{name}.csv") for name in present]
    rows_count = len(options.environments)
    with paper_style():
        figure, axes = plt.subplots(
            rows_count, 3, figsize=(FULL_WIDTH, 3.0 * rows_count + 0.9), squeeze=False
        )
        for index, environment in enumerate(options.environments):
            tables = data[environment]
            draw_profile(axes[index][0], environment, tables["profile"])
            draw_dose(
                axes[index][1],
                environment,
                tables["dose"],
                reference_level(tables["readings"]),
            )
            draw_readings(axes[index][2], environment, tables["readings"])
        figure.legend(
            handles=legend_items(options.environments[0]),
            loc="upper center",
            ncol=4,
            frameon=False,
            bbox_to_anchor=(0.5, 1.0),
        )
        figure.tight_layout(rect=(0, 0, 1, 1.0 - 0.9 / (3.0 * rows_count + 0.9)))
    panels = {}
    for environment in options.environments:
        tables = data[environment]
        panels[f"{environment}_profile"] = standalone(
            lambda ax, e=environment, t=tables: draw_profile(ax, e, t["profile"]),
            size=PANEL,
        )
        panels[f"{environment}_dose"] = standalone(
            lambda ax, e=environment, t=tables: draw_dose(
                ax, e, t["dose"], reference_level(t["readings"])
            ),
            size=PANEL,
        )
    tables_out: dict[str, list[Row]] = {}
    for kind in ("readings", "dose"):
        tables_out[kind] = [
            {"environment": e, **row}
            for e in options.environments
            for row in data[e][kind]
        ]
    return BuiltFigure(
        "figure_attention_summary", figure, tables_out, notes, sources, panels
    )

In [ ]:
# Parameters: `environments` the benchmarks drawn (the declaration's two),
# `split` the roster the attention files were written for.
options = AttentionOptions()
attention = build_attention_figure(layout, options)
display(attention.figure)

In [ ]:
# What was read per benchmark, and what is pending.
print(attention.notes)

In [ ]:
# The declared readings as read (estimate and joint seed/task interval).
display(Markdown(markdown_preview(attention.tables["readings"], limit=24)))

In [ ]:
# Save: the composite (PDF + PNG), every panel (PNG), the tables and the notes.
for path in attention.save(FIGURES_DIR, DATA_DIR):
    print(path.relative_to(REPO) if path.is_relative_to(REPO) else path)
attention.close()